# Continuous Perfusion Operating Mode Test Notebook

This notebook tests the **CONTINUOUS_PERFUSION** operating mode implementation.

**Test Coverage:**
- Simulation setup and execution
- Single batch run with initial conditions
- Multi-batch run with auto-extracted initial conditions
- Data integrity validation
- Performance metrics
- Continuous perfusion-specific validations (continuous feed + perfusion, steady-state)

## 1. Setup and Imports

In [1]:
import sys
import numpy as np
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.core.parameters import SimulationParameters
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.stoichiometry import StoichiometryMode
from animakinetica.models.odesystems import OperatingMode
from animakinetica.models.perfusion_strategies import PerfusionStrategy
from animakinetica.models.gas_controllers import GasController
from animakinetica.models.feed_strategies import FeedStrategy
print("✓ Imports successful")

✓ Imports successful


## 2. Load Parameters and Initialize Simulation

In [2]:
# Load base parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "excess_kinetics_adjusted", "fed_batch","perfusion", "HEPES"])

# Initialize simulation with CONTINUOUS_PERFUSION mode
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.EXCESS,
    operating_mode=OperatingMode.CONTINUOUS_PERFUSION,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    perfusion_strategy=PerfusionStrategy.CSPR,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=False,  # Store detailed data for testing
    solver_method="BDF"
)

print(f"✓ Simulation initialized")
print(f"  Operating Mode: {sim.operating_mode}")
print(f"  Kinetics Model: {sim.kinetics}")
print(f"  Perfusion Strategy: {sim.perfusion_strategy}")
print(f"  Gas Controller: {sim.gas_controller}")
print(f"  Bleed Rate (B): {sim.parameters.perfusion.B.value:.4f} 1/h")

Loading parameters from 8 file(s) sequentially...
  [1/8] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/8] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/8] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/8] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/8] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\excess_kinetics\excess_kinetics_adjusted.yaml
  [6/8] Updating with: c:\Users\btumulero\OneDrive - Delft University of Tec

## 4. Multi-Batch Run (Auto-Extracted Initial Conditions)

In [3]:

# Run 5 repeated batches
num_batches = 5
multi_result = sim.run_simulation(number_of_batches=num_batches)

print(f"✓ {num_batches} batches completed")
print(f"\nSummary Statistics:")
print(f"  Average batch duration: {multi_result.projected_totals_stats['total_simulation_time_h']/num_batches:.2f} h")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")

print(f"\nBatch-by-Batch Results:")
for i, batch in enumerate(sim.data.batches, 1):
    print(f"  Batch {i}: {batch.t_end:.2f} h, Xv={batch.Xv_final:.2e} cells/L, V={batch.V_final:.2f} L")


run_simulation execution time: 2290.544 ms
✓ 5 batches completed

Summary Statistics:
  Average batch duration: 960.00 h
  STY: 2.886 kg/m3/day

Batch-by-Batch Results:
  Batch 1: 960.00 h, Xv=6.78e+10 cells/L, V=2000.00 L
  Batch 2: 960.00 h, Xv=6.78e+10 cells/L, V=2000.00 L
  Batch 3: 960.00 h, Xv=6.78e+10 cells/L, V=2000.00 L
  Batch 4: 960.00 h, Xv=6.78e+10 cells/L, V=2000.00 L
  Batch 5: 960.00 h, Xv=6.78e+10 cells/L, V=2000.00 L


## 5. Data Integrity Validation

In [4]:
# Test batch 1 data integrity
batch_1 = sim.data.batches[0]

print("Data Integrity Checks for Batch 1:")
print("\n1. State Variables:")
# y_points shape: (n_states, n_timepoints) - states are rows
# [Xv, C_Glc, C_Gln, C_Lac, C_Amm, V, C_O2, C_CT, C_H, C_Na, C_AA, C_Osm_bg]
state_vars = ['Xv', 'C_Glc', 'C_Gln', 'C_Lac', 'C_Amm', 'V', 'C_O2', 'C_CT', 'C_H', 'C_Na', 'C_AA', 'C_Osm_bg']
for idx, var in enumerate(state_vars):
    data = np.asarray(batch_1.y_points[idx], dtype=float)  # Ensure numeric array
    print(f"  {var}: {len(data)} points, range [{data.min():.2e}, {data.max():.2e}]")
    assert len(data) > 0, f"{var} data is empty"
    assert not np.any(np.isnan(data)), f"{var} contains NaN values"

print("\n2. Kinetic Rates:")
kinetic_rates = ['mu', 'q_Glc', 'q_Gln', 'q_Lac', 'q_Amm', 'q_AA', 'q_O2', 'q_CO2']
for rate in kinetic_rates:
    rate_list = getattr(batch_1, f'{rate}_list')
    print(f"  {rate}: {len(rate_list)} points, range [{rate_list.min():.2e}, {rate_list.max():.2e}]")
    assert len(rate_list) > 0, f"{rate}_list is empty"
    assert not np.any(np.isnan(rate_list)), f"{rate}_list contains NaN values"

print("\n3. Nutrient Factors:")
nf_vars = ['NF_glucose', 'NF_glutamine', 'NF_O2', 'NF']
for nf in nf_vars:
    nf_list = getattr(batch_1, f'{nf}_list')
    print(f"  {nf}: {len(nf_list)} points, range [{nf_list.min():.2e}, {nf_list.max():.2e}]")
    assert len(nf_list) > 0, f"{nf}_list is empty"
    assert np.all((nf_list >= 0) & (nf_list <= 1)), f"{nf}_list contains values outside [0,1]"

print("\n4. Inhibition Factors:")
if_vars = ['IF_O2', 'IF_CO2', 'IF_lactate', 'IF_ammonia', 'IF_osmolality', 'IF']
for ifvar in if_vars:
    if_list = getattr(batch_1, f'{ifvar}_list')
    print(f"  {ifvar}: {len(if_list)} points, range [{if_list.min():.2e}, {if_list.max():.2e}]")
    assert len(if_list) > 0, f"{ifvar}_list is empty"
    assert np.all((if_list >= 0) & (if_list <= 1)), f"{ifvar}_list contains values outside [0,1]"

print("\n5. Gas Control Variables:")
gas_vars = ['N', 'Fg', 'y_O2', 'kLa', 'OTR', 'CTR', 'C_CO2']
for gv in gas_vars:
    gv_list = getattr(batch_1, f'{gv}_list')
    print(f"  {gv}: {len(gv_list)} points, range [{gv_list.min():.2e}, {gv_list.max():.2e}]")
    assert len(gv_list) > 0, f"{gv}_list is empty"
    assert not np.any(np.isnan(gv_list)), f"{gv}_list contains NaN values"

print("\n6. pH Control Variables:")
print(f"  D_base: {len(batch_1.D_base_list)} points, range [{batch_1.D_base_list.min():.2e}, {batch_1.D_base_list.max():.2e}]")
assert len(batch_1.D_base_list) > 0, "D_base_list is empty"

print("\n7. Dilution Rate (Continuous Perfusion-specific):")
D_list = batch_1.D_list
print(f"  D: {len(D_list)} points, range [{D_list.min():.4f}, {D_list.max():.4f}] 1/h")
assert len(D_list) > 0, "D_list is empty"
assert not np.any(np.isnan(D_list)), "D_list contains NaN values"

print("\n✓ All data integrity checks passed")


Data Integrity Checks for Batch 1:

1. State Variables:
  Xv: 1023 points, range [6.00e+08, 6.78e+10]
  C_Glc: 1023 points, range [8.18e-01, 2.50e+01]
  C_Gln: 1023 points, range [1.94e+00, 1.00e+01]
  C_Lac: 1023 points, range [0.00e+00, 9.09e+00]
  C_Amm: 1023 points, range [0.00e+00, 2.73e+00]
  V: 1023 points, range [6.00e+02, 2.00e+03]
  C_O2: 1023 points, range [1.00e-02, 1.00e-02]
  C_CT: 1023 points, range [1.87e+01, 2.90e+01]
  C_H: 1023 points, range [6.31e-08, 6.31e-08]
  C_Na: 1023 points, range [0.00e+00, 0.00e+00]
  C_AA: 1023 points, range [4.98e-01, 7.72e+00]
  C_Osm_bg: 1023 points, range [2.50e+02, 2.50e+02]

2. Kinetic Rates:
  mu: 1023 points, range [1.69e-02, 2.80e-02]
  q_Glc: 1023 points, range [5.66e-11, 6.25e-11]
  q_Gln: 1023 points, range [8.46e-12, 1.34e-11]
  q_Lac: 1023 points, range [3.48e-12, 3.41e-11]
  q_Amm: 1023 points, range [4.47e-12, 8.83e-12]
  q_AA: 1023 points, range [8.61e-12, 1.43e-11]
  q_O2: 1023 points, range [2.00e-10, 2.93e-10]
  q_CO2: 

## 6. Performance Metrics

In [5]:
print("Performance Metrics (5-Batches):")
print(f"\n Total Biomass: {multi_result.projected_totals_stats['total_DCM_out_kg']:.2f} kg")
print(f"  STY: {multi_result.summary_stats['STY_kg_m3_day']:.3f} kg/m3/day")
print(f"  Medium Conversion Efficiency: {multi_result.summary_stats['medium_conversion_efficiency_m3_kg']:.3f} m3/kg DCM")
print(f"  Protein Conversion Efficiency: {multi_result.summary_stats['protein_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Feed Conversion Efficiency: {multi_result.summary_stats['feed_conversion_efficiency_kg_kg']:.3f} kg/kg")
print(f"  Total Simulation Time: {multi_result.projected_totals_stats['total_simulation_time_h']:.2f} h")
print(f"  Number of Batches: {multi_result.projected_totals_stats['number_of_batches']}")


# Validate metrics are reasonable
assert multi_result.projected_totals_stats['total_DCM_out_kg'] > 0, "Total DCM should be positive"
assert multi_result.summary_stats['STY_kg_m3_day'] > 0, "STY should be positive"
assert 0 < multi_result.summary_stats['medium_conversion_efficiency_m3_kg'] < 1000, "Medium conversion efficiency should be positive and reasonable"
assert multi_result.summary_stats['protein_conversion_efficiency_kg_kg'] > 0, "Protein conversion efficiency should be positive"
assert multi_result.summary_stats['feed_conversion_efficiency_kg_kg'] > 0, "Feed conversion efficiency should be positive"
assert multi_result.projected_totals_stats['total_simulation_time_h'] > 0, "Total simulation time should be positive"
assert multi_result.projected_totals_stats['number_of_batches'] == num_batches, f"Number of batches should be {num_batches}"

print("\n✓ Performance metrics validated")


Performance Metrics (5-Batches):

 Total Biomass: 1281.64 kg
  STY: 2.886 kg/m3/day
  Medium Conversion Efficiency: 1.309 m3/kg DCM
  Protein Conversion Efficiency: 3.915 kg/kg
  Feed Conversion Efficiency: 7.153 kg/kg
  Total Simulation Time: 4800.00 h
  Number of Batches: 5

✓ Performance metrics validated


In [6]:
sim.data.plot_biomass_metabolites()

In [7]:
sim.data.plot_volume_feed()

In [8]:
sim.data.plot_pH_control(batch_number=1)

In [9]:
sim.data.plot_gas_transfer()

In [10]:
sim.data.plot_kinetics()

In [11]:
sim.data.plot_cumulative_harvest()

## 7. Continuous Perfusion-Specific Validations

In [12]:
print("Continuous Perfusion-Specific Validations:")

# 1. Volume control (should remain approximately constant)
print("\n1. Volume Control:")
for i, batch in enumerate(sim.data.batches, 1):
    V_initial = batch.y_points[5, 0]
    V_final = batch.y_points[5, -1]
    V_max = batch.y_points[5, :].max()
    V_mean = batch.y_points[5, :].mean()
    V_std = batch.y_points[5, :].std()
    print(f"  Batch {i}: V_mean={V_mean:.2f} L, V_std={V_std:.3f} L (CV={V_std/V_mean*100:.2f}%)")
    print(f"           V_initial={V_initial:.2f} L, V_final={V_final:.2f} L, V_max={V_max:.2f} L")
    print(f" Termination reason: {batch.termination_reason}")

# 2. Dilution rate profile (combination of feeding + perfusion)
print("\n2. Dilution Rate Profile (Batch 1):")
D_mean = batch_1.D_list.mean()
D_std = batch_1.D_list.std()
D_min = batch_1.D_list.min()
D_max = batch_1.D_list.max()
print(f"  D_mean: {D_mean:.4f} 1/h")
print(f"  D_std: {D_std:.4f} 1/h")
print(f"  D_range: [{D_min:.4f}, {D_max:.4f}] 1/h")
print(f"  Coefficient of variation: {D_std/D_mean*100:.2f}%")

# 3. Continuous feed component analysis
print("\n3. Continuous Feed Analysis:")
# In continuous perfusion, both feed and harvest occur simultaneously
# The parameter 'B' determines feed contribution to dilution rate
B = float(params.perfusion.B) if params.perfusion.B is not None else 0.0
D_base = float(params.perfusion.D_baseline)
print(f"  Feed contribution parameter (B): {B:.4f}")
print(f"  Base perfusion rate (D_base): {D_base:.4f} 1/h")
print(f"  Expected total D ≈ D_perfusion + B (if applicable)")

# 4. Steady-state convergence
print("\n4. Steady-State Convergence (Batch 1):")
# Analyze last 20% of run
n_points = len(batch_1.t_points)
ss_start_idx = int(0.8 * n_points)

state_indices = {'Xv': 0, 'C_Glc': 1, 'C_Gln': 2, 'C_Lac': 3, 'C_Amm': 4 , 'C_AA': 10}
for var_name, idx in state_indices.items():
    data_ss = batch_1.y_points[idx, ss_start_idx:]
    mean_ss = data_ss.mean()
    std_ss = data_ss.std()
    cv_ss = (std_ss / mean_ss) * 100 if mean_ss > 0 else 0
    print(f"  {var_name}: mean={mean_ss:.2e}, std={std_ss:.2e}, CV={cv_ss:.2f}%")
    if cv_ss < 5.0:
        print(f"    ✓ Reached steady-state (CV < 5%)")
    else:
        print(f"    → Still converging (CV ≥ 5%)")

# 5. Growth rate vs dilution rate balance
print("\n5. Growth Rate vs Dilution Rate Balance:")
for i, batch in enumerate(sim.data.batches, 1):
    mu_mean = batch.mu_list[ss_start_idx:].mean()
    D_mean_ss = batch.D_list[ss_start_idx:].mean()
    print(f"  Batch {i}:")
    print(f"    mu_mean (steady-state): {mu_mean:.4f} 1/h")
    print(f"    D_mean (steady-state): {D_mean_ss:.4f} 1/h")
    print(f"    Net growth (mu - D): {mu_mean - D_mean_ss:.4f} 1/h")
    print(f"    mu/D ratio: {mu_mean/D_mean_ss:.3f}")
    # In steady-state, net growth should be close to 0
    # But cell retention means some cells are harvested, so mu > D_perfusion

# 6. Cell retention and productivity
print("\n6. Cell Retention and Productivity:")
eff_separator = float(params.perfusion.eff_separator)
for i, batch in enumerate(sim.data.batches, 1):
    Xv_initial = batch.y_points[0, 0]
    Xv_final = batch.y_points[0, -1]
    Xv_max = batch.y_points[0, :].max()
    Xv_mean_ss = batch.y_points[0, ss_start_idx:].mean()
    print(f"  Batch {i}:")
    print(f"    Xv_initial: {Xv_initial:.2e} cells/L")
    print(f"    Xv_max: {Xv_max:.2e} cells/L")
    print(f"    Xv_mean (steady-state): {Xv_mean_ss:.2e} cells/L")
    print(f"    Cell retention efficiency: {eff_separator*100:.1f}%")

# 7. Metabolite control in steady-state
print("\n7. Metabolite Control in Steady-State:")
for i, batch in enumerate(sim.data.batches, 1):
    C_Lac_mean_ss = batch.y_points[3, ss_start_idx:].mean()
    C_Amm_mean_ss = batch.y_points[4, ss_start_idx:].mean()
    C_Lac_max = batch.y_points[3, :].max()
    C_Amm_max = batch.y_points[4, :].max()
    print(f"  Batch {i}:")
    print(f"    Lactate: mean_ss={C_Lac_mean_ss:.2f}, max={C_Lac_max:.2f} mmol/L")
    print(f"    Ammonia: mean_ss={C_Amm_mean_ss:.2f}, max={C_Amm_max:.2f} mmol/L")

# 8. Substrate levels in steady-state
print("\n8. Substrate Levels in Steady-State:")
C_Glc_feed = float(params.perfusion.C_Glc_feed)
C_Gln_feed = float(params.perfusion.C_Gln_feed)
for i, batch in enumerate(sim.data.batches, 1):
    C_Glc_mean_ss = batch.y_points[1, ss_start_idx:].mean()
    C_Gln_mean_ss = batch.y_points[2, ss_start_idx:].mean()
    print(f"  Batch {i}:")
    print(f"    Glucose: mean_ss={C_Glc_mean_ss:.2f} mmol/L (feed={C_Glc_feed:.2f}, utilization={(1-C_Glc_mean_ss/C_Glc_feed)*100:.1f}%)")
    print(f"    Glutamine: mean_ss={C_Gln_mean_ss:.2f} mmol/L (feed={C_Gln_feed:.2f}, utilization={(1-C_Gln_mean_ss/C_Gln_feed)*100:.1f}%)")

# 9. Comparison with regular perfusion
print("\n9. Comparison with Concentrated Fedbatch:")
params.update("fed_batch")
print("  Testing Concentrated Fedbatch for comparison...")
sim_p = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    operating_mode=OperatingMode.CONCENTRATED_FEDBATCH,
    feed_strategy= FeedStrategy.EXPONENTIAL,
    perfusion_strategy=PerfusionStrategy.CONSTANT,
    gas_controller=GasController.CASCADE_N_FG_XO2,
    performance_mode=True
)
result_p = sim_p.run_simulation()
batch_results_p = sim_p.data.batches[0]
print(f"    Concentrated fedbatch: Xv_final={batch_results_p.Xv_final:.2e} cells/L, t_end={batch_results_p.t_end:.1f} h")
print(f"    Continuous perfusion: Xv_final={batch_1.Xv_final:.2e} cells/L, t_end={batch_1.t_end:.1f} h")

print("\n✓ All continuous perfusion-specific validations completed")

Continuous Perfusion-Specific Validations:

1. Volume Control:
  Batch 1: V_mean=1798.98 L, V_std=475.753 L (CV=26.45%)
           V_initial=600.00 L, V_final=2000.00 L, V_max=2000.00 L
 Termination reason: Maximum time reached
  Batch 2: V_mean=1798.98 L, V_std=475.753 L (CV=26.45%)
           V_initial=600.00 L, V_final=2000.00 L, V_max=2000.00 L
 Termination reason: Maximum time reached
  Batch 3: V_mean=1798.98 L, V_std=475.753 L (CV=26.45%)
           V_initial=600.00 L, V_final=2000.00 L, V_max=2000.00 L
 Termination reason: Maximum time reached
  Batch 4: V_mean=1798.98 L, V_std=475.753 L (CV=26.45%)
           V_initial=600.00 L, V_final=2000.00 L, V_max=2000.00 L
 Termination reason: Maximum time reached
  Batch 5: V_mean=1798.98 L, V_std=475.753 L (CV=26.45%)
           V_initial=600.00 L, V_final=2000.00 L, V_max=2000.00 L
 Termination reason: Maximum time reached

2. Dilution Rate Profile (Batch 1):
  D_mean: 0.1529 1/h
  D_std: 0.0755 1/h
  D_range: [0.0000, 0.2000] 1/h
  

## 8. Feed rates and glucose concentration visualizations

In [13]:
# Comparison plot: perfusion profiles, glucose concentration, and biomass across all perfusion strategies
import copy
import traceback
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from animakinetica.analysis.batch_visualization import (
    plot_batch_volume_feed,
    plot_batch_biomass_metabolites,
)

# Use same simulation setup for fair comparison
perfusion_strategies_compare = [
    PerfusionStrategy.CONSTANT,
    PerfusionStrategy.TOXICITY_BASED_STEP,
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL,
    PerfusionStrategy.CSPR,
    PerfusionStrategy.GLUCOSE_SETPOINT,
]

color_map = {
    PerfusionStrategy.CONSTANT: "#1f77b4",
    PerfusionStrategy.TOXICITY_BASED_STEP: "#ff7f0e",
    PerfusionStrategy.TOXICITY_BASED_PROPORTIONAL: "#2ca02c",
    PerfusionStrategy.CSPR: "#d62728",
    PerfusionStrategy.GLUCOSE_SETPOINT: "#9467bd",
}

fig_compare = make_subplots(
    rows=1, cols=6,
    subplot_titles=(
        "Process Perfusion Rate Comparison",
        "Base Feed Rate Comparison",
        "Total Inlet Rate Comparison",
        "Volume Comparison",
        "Glucose Comparison",
        "Biomass Comparison",
    ),
    horizontal_spacing=0.06
)

params_compare = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch", "perfusion"])
,
def _build_initial_conditions(sim_obj):
    base = sim_obj.base
    return [
        base['X_init'],
        base['C_Glc_init'],
        base['C_Gln_init'],
        base['C_Lac_init'],
        base['C_Amm_init'],
        base['V_init'],
        base['C_O2_init'],
        base['C_CT_init'],
        base['C_H_init'],
        base['C_Na_init'],
        base['C_AA_init'],
    ]

def _print_event_snapshot(sim_obj, y0):
    print("  Event snapshots at t=0 (event(t=0, y0, params)):")
    for phase_name, event_list in sim_obj._phase_events.items():
        print(f"    Phase: {phase_name}")
        for event_fn in event_list:
            try:
                event_value = float(event_fn(0.0, y0, sim_obj.parameters))
            except Exception as event_exc:
                print(f"      {event_fn.__name__}: ERROR evaluating event -> {event_exc}")
                continue
            direction = getattr(event_fn, "direction", None)
            terminal = getattr(event_fn, "terminal", None)
            print(f"      {event_fn.__name__}: value={event_value:.6e}, direction={direction}, terminal={terminal}")

failed_diagnostics = []

for ps in perfusion_strategies_compare:
    params_test = copy.deepcopy(params_compare)
    params_test.update("excess_kinetics_batch")  # Use excess kinetics for more dynamic profiles
    sim_test = CellCultureSimulation(
        parameters=params_test,
        kinetics=Kinetics.EXCESS,
        stoichiometry_mode=StoichiometryMode.DYNAMIC,
        feed_strategy=FeedStrategy.EXPONENTIAL,
        operating_mode=OperatingMode.CONTINUOUS_PERFUSION,
        perfusion_strategy=ps,
        gas_controller=GasController.CASCADE_N_FG_XO2,
        performance_mode=False,
        solver_method="BDF"
    )

    try:
        res = sim_test.run_simulation(number_of_batches=1)
    except ValueError as exc:
        print(f"\n[DIAGNOSTIC] Strategy {ps.name} failed with ValueError: {exc}")
        y0 = _build_initial_conditions(sim_test)
        _print_event_snapshot(sim_test, y0)
        failed_diagnostics.append(
            {
                "strategy": ps.name,
                "error": str(exc),
                "traceback": traceback.format_exc(),
            }
        )
        continue

    # Reuse existing plotting utilities and extract traces from their outputs
    fig_volume = plot_batch_volume_feed(res, batch_number=1, layout_kwargs={"showlegend": False})
    fig_biomass = plot_batch_biomass_metabolites(res, batch_number=1, layout_kwargs={"showlegend": False})

    process_rate_trace = next(tr for tr in fig_volume.data if tr.name == "Process Feed Rate")
    base_rate_trace = next(tr for tr in fig_volume.data if tr.name == "Base Feed Rate")
    total_rate_trace = next(tr for tr in fig_volume.data if tr.name == "Total Inlet Rate")
    volume_trace = next(tr for tr in fig_volume.data if tr.name == "Volume")
    glucose_trace = next(tr for tr in fig_biomass.data if tr.name == "Glucose")
    biomass_trace = next(tr for tr in fig_biomass.data if tr.name == "Cell Density")

    label = ps.name
    color = color_map.get(ps, None)

    fig_compare.add_trace(
        go.Scatter(
            x=process_rate_trace.x,
            y=process_rate_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=True,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_process: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=1,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=base_rate_trace.x,
            y=base_rate_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_base: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=2,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=total_rate_trace.x,
            y=total_rate_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>F_total: %{y:.4f} L/h<extra></extra>",
        ),
        row=1, col=3,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=volume_trace.x,
            y=volume_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color, dash="dot"),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>V: %{y:.2f} L<extra></extra>",
        ),
        row=1, col=4,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=glucose_trace.x,
            y=glucose_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>Glucose: %{y:.2f} mmol/L<extra></extra>",
        ),
        row=1, col=5,
    )

    fig_compare.add_trace(
        go.Scatter(
            x=biomass_trace.x,
            y=biomass_trace.y,
            mode="lines",
            name=label,
            line=dict(width=2, color=color, dash="dash"),
            showlegend=False,
            hovertemplate="Strategy: " + label + "<br>Time: %{x:.1f} h<br>Biomass: %{y:.2f} 10⁶ cells/mL<extra></extra>",
        ),
        row=1, col=6,
    )

for col in range(1, 7):
    fig_compare.update_xaxes(title_text="Time (h)", row=1, col=col)

fig_compare.update_yaxes(title_text="Process Perfusion Rate (L/h)", row=1, col=1)
fig_compare.update_yaxes(title_text="Base Feed Rate (L/h)", row=1, col=2)
fig_compare.update_yaxes(title_text="Total Inlet Rate (L/h)", row=1, col=3)
fig_compare.update_yaxes(title_text="Volume (L)", row=1, col=4)
fig_compare.update_yaxes(title_text="Glucose (mmol/L)", row=1, col=5)
fig_compare.update_yaxes(title_text="10⁶ cells/mL", row=1, col=6)

fig_compare.update_layout(
    title_text="Perfusion Strategy Comparison",
    template="plotly_white",
    height=500,
    width=3800,
    legend=dict(x=0.01, y=0.99, bgcolor="rgba(255,255,255,0.8)"),
)

fig_compare.show()

if failed_diagnostics:
    print("\nFailed strategies summary:")
    for item in failed_diagnostics:
        print(f"  - {item['strategy']}: {item['error']}")
else:
    print("\nAll strategies completed without ValueError.")

Loading parameters from 6 file(s) sequentially...
  [1/6] Base: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
  [6/6] Updating with: c:\Users\btumulero\OneDrive - Delft University of Technology\2. Code


All strategies completed without ValueError.
